# Importing Important Modules

In [1]:
import sys
from pathlib import Path

# Add the project root (parent of notebooks) to Python's search path
sys.path.append(str(Path().resolve().parent))

import os
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from vanilla_bsf import build_model

# Configuration

In [2]:
ACTIVATION_FOLDER = r"C:\Users\97433\BSF_implementation\data\rabit_activations"
INPUT_DIM = 768
NUM_BLOCKS = 4096
BLOCK_SIZE = 4
TOP_K = 16
BATCH_SIZE = 4096
LR = 1e-4
EPOCHS = 300
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Dataset

In [3]:
class PatchDataset(Dataset):

    def __init__(self, folder):

        self.samples = []

        files = sorted(os.listdir(folder))

        for file in tqdm(files):

            if not file.endswith(".npy"):
                continue

            acts = np.load(os.path.join(folder, file))

            # acts = (256,768)

            self.samples.append(acts)

        #####################################
        # Concatenate all patches
        #####################################

        self.samples = np.concatenate(
            self.samples,
            axis=0,
        )

        print(self.samples.shape)

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, idx):

        x = self.samples[idx]

        return torch.from_numpy(x).float()

# DataLoader

In [4]:
dataset = PatchDataset(ACTIVATION_FOLDER)

train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)

100%|██████████| 300/300 [00:02<00:00, 111.42it/s]


(76800, 768)


# Model

In [5]:
model = build_model(
    input_dim=INPUT_DIM,
    num_blocks=NUM_BLOCKS,
    block_size=BLOCK_SIZE,
    top_k=TOP_K,
)

model.to(DEVICE)

Vanilla_BSF(
  (W): Linear(in_features=768, out_features=16384, bias=False)
  (D): Linear(in_features=16384, out_features=768, bias=False)
)

# Optimizer

In [6]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
)

criterion = nn.MSELoss()

# Training Loop

In [7]:
history = []

best_loss = float("inf")

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0

    dead_block_counter = None

    for x in tqdm(train_loader):

        x = x.to(DEVICE)

        output = model(x)

        reconstruction = output["reconstruction"]

        loss = criterion(
            reconstruction,
            x,
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        ###################################
        # Dead blocks
        ###################################

        mask = output["active_mask"]

        if dead_block_counter is None:

            dead_block_counter = mask.sum(0)

        else:

            dead_block_counter += mask.sum(0)

    epoch_loss = running_loss / len(train_loader)

    history.append(epoch_loss)

    dead_blocks = (dead_block_counter == 0).sum().item()

    print(
        f"Epoch {epoch+1:03d}"
        f" | Loss = {epoch_loss:.6f}"
        f" | Dead Blocks = {dead_blocks}"
    )

    if epoch_loss < best_loss:

        best_loss = epoch_loss

        torch.save(
            model.state_dict(),
            "best_model.pth",
        )

100%|██████████| 18/18 [02:53<00:00,  9.65s/it]


Epoch 001 | Loss = 2.913275 | Dead Blocks = 0


100%|██████████| 18/18 [03:25<00:00, 11.40s/it]


Epoch 002 | Loss = 2.315498 | Dead Blocks = 56


100%|██████████| 18/18 [02:56<00:00,  9.82s/it]


...

Epoch 024 | Loss = 0.519111 | Dead Blocks = 1032


100%|██████████| 18/18 [04:04<00:00, 13.59s/it]


Epoch 025 | Loss = 0.511492 | Dead Blocks = 1032


 78%|███████▊  | 14/18 [03:51<01:06, 16.55s/it]


KeyboardInterrupt: 

# Reconstruction Test

In [ ]:
model.eval()

x = dataset[0].unsqueeze(0).to(DEVICE)

with torch.no_grad():

    output = model(x)

print("Input shape")

print(x.shape)

print()

print("Reconstruction shape")

print(output["reconstruction"].shape)

print()

print("Latent shape")

print(output["z"].shape)

print()

print("Block shape")

print(output["z_blocks"].shape)